In [1]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

In [2]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("Torch version:", torch.__version__)

CUDA available: True
CUDA device name: NVIDIA GeForce RTX 4080 SUPER
CUDA version: 12.1
Torch version: 2.5.1+cu121


In [3]:
#!pip install mmengine

In [4]:
#!pip install lightning-template

In [5]:
sys.path.append(os.path.abspath('../'))  # Adjust this path based on where your files are located
print(sys.path)

from datasets.hint.hint_admet import HINTDataset 
from datasets.ctod.ctod_enrollment import CTODataset
from mmcto.mmcto import MMCTO
from mmcto.layers.sparse_moe import SparseMOELayer, FeedForwardLayer
from mmf.early_fusion import EarlyFusion
from mmf.middle_fusion import MiddleFusion
from mmf.late_fusion import LateFusion
from torch.utils.data import Dataset
import pytorch_lightning.callbacks as pl_callbacks

['C:\\Users\\Carol\\Documents\\Data Code and Deliverables\\Data Group Part\\project\\models', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\python311.zip', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\DLLs', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis', '', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages\\win32', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages\\Pythonwin', 'C:\\Users\\Carol\\Documents\\Data Code and Deliverables\\Data Group Part\\project']


In [6]:
PHASE = "III"
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIM = 768

In [7]:
#base_path = r"C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint"
base_path = r"C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\ctod"
#admet_path = os.path.join(base_path, "Admet", "cooked")

data_prefix = dict(
    data_path=base_path,
    table_path=os.path.join(base_path, "text_description"),
    summarization_path=os.path.join(base_path, "brief_summary"),
    drug_description_path=os.path.join(base_path, "drugbank", "druginfo_description.json"),
    criteria_path=os.path.join(base_path, "criteria"),
    #admet_path=admet_path,
)

In [8]:
def move_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    elif isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [move_to_device(v, device) for v in obj]
    return obj

In [9]:
def build_encoder():
    return nn.TransformerEncoder(
        nn.TransformerEncoderLayer(
            d_model=MODEL_DIM,
            nhead=8,
            dim_feedforward=2048,
            dropout=0.1,
            activation='relu',
            batch_first=True
        ),
        num_layers=2
    )
input_parts = [
    "table", "summarization", "description", "criteria",
    "smiles", "smiles_concat", "smiles_summarization",
    "drugs", "drugs_concat", "drugs_summarization",
    "diseases", "diseases_concat", "diseases_summarization",
    "smiles_transformer_concat", "enrollment"
]

encoders = nn.ModuleDict({k: build_encoder() for k in input_parts})
smoe_encoder = SparseMOELayer(
    expert_cfg=FeedForwardLayer,
    num_experts=4,
    input_dim=MODEL_DIM,
    topk=2,
)

In [10]:
model = MMCTO(
    encoders=encoders,
    smoe_encoder=smoe_encoder,
    aux_loss=True,
    aux_loss_share_fc=False,
    moe_method="weighted",
    vocab_size=28996,
    model_dim=MODEL_DIM,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [11]:
# train_dataset = HINTDataset(
#     data_prefix=data_prefix,
#     ann_file_name=f"phase_{PHASE}_train",
#     serialize_data=False,
# )
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     collate_fn=train_dataset.collate_fn,
# )

# test_dataset = HINTDataset(
#     data_prefix=data_prefix,
#     ann_file_name=f"phase_{PHASE}_test",
#     serialize_data=False,
# )
# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     collate_fn=test_dataset.collate_fn,
# )

[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint\phase_III_train_tokenized.pt
[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint\phase_III_test_tokenized.pt


In [11]:
train_dataset = CTODataset(
    data_prefix=data_prefix,
    ann_file_name=f"phase_{PHASE}_train",
    serialize_data=False,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
)

test_dataset = CTODataset(
    data_prefix=data_prefix,
    ann_file_name=f"phase_{PHASE}_valid",
    serialize_data=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=test_dataset.collate_fn,
)

[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\ctod\phase_III_train_tokenized.pt
[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\ctod\phase_III_valid_tokenized.pt


In [9]:
param_device = next(model.parameters()).device
print(f"[INFO] Model is on device: {param_device}")

# Confirm GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**2  # in MB
    reserved = torch.cuda.memory_reserved() / 1024**2    # in MB
    print(f"[INFO] GPU Memory - Allocated: {allocated:.2f} MB | Reserved: {reserved:.2f} MB")
else:
    print("[WARNING] CUDA is not available. Using CPU.")

[INFO] Model is on device: cuda:0
[INFO] GPU Memory - Allocated: 671.15 MB | Reserved: 722.00 MB


## HINT

### PHASE I

In [16]:
print(f"\nTraining MMCTO on Phase {PHASE}...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")



Training MMCTO on Phase I...

Epoch 1 | Avg Loss: 0.4054
Epoch 2 | Avg Loss: 0.3879
Epoch 3 | Avg Loss: 0.3848
Epoch 4 | Avg Loss: 0.3721
Epoch 5 | Avg Loss: 0.3561
Epoch 6 | Avg Loss: 0.3333
Epoch 7 | Avg Loss: 0.2899
Epoch 8 | Avg Loss: 0.2512
Epoch 9 | Avg Loss: 0.1990
Epoch 10 | Avg Loss: 0.1514
Epoch 11 | Avg Loss: 0.1041
Epoch 12 | Avg Loss: 0.0849
Epoch 13 | Avg Loss: 0.0575
Epoch 14 | Avg Loss: 0.0454
Epoch 15 | Avg Loss: 0.0346
Epoch 16 | Avg Loss: 0.0342
Epoch 17 | Avg Loss: 0.0222
Epoch 18 | Avg Loss: 0.0290
Epoch 19 | Avg Loss: 0.0146
Epoch 20 | Avg Loss: 0.0090


In [17]:
torch.save(model.state_dict(), "mmcto_phaseI_gaiting.pth")

In [18]:
print(f"\nEvaluating on Phase {PHASE} Test Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase I Test Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [19]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      75.64  28.37  45.77
summarization              87.60  83.58  68.06
smiles                     79.25   0.00  52.52
description                79.29  85.92  56.55
criteria                   77.80  75.78  51.11
enrollment                 77.49  87.32  50.00
diseases                   66.31  19.36  30.08
drugs                      75.81  20.74  45.89
All (Fused)                83.45  87.11  64.12


## PHASE II

In [9]:
print(f"\nTraining MMCTO on Phase {PHASE}...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")



Training MMCTO on Phase II...

Epoch 1 | Avg Loss: 0.5088
Epoch 2 | Avg Loss: 0.5014
Epoch 3 | Avg Loss: 0.4946
Epoch 4 | Avg Loss: 0.4862
Epoch 5 | Avg Loss: 0.4726
Epoch 6 | Avg Loss: 0.4493
Epoch 7 | Avg Loss: 0.4134
Epoch 8 | Avg Loss: 0.3657
Epoch 9 | Avg Loss: 0.3043
Epoch 10 | Avg Loss: 0.2516
Epoch 11 | Avg Loss: 0.1930
Epoch 12 | Avg Loss: 0.1477
Epoch 13 | Avg Loss: 0.1104
Epoch 14 | Avg Loss: 0.1031
Epoch 15 | Avg Loss: 0.0750
Epoch 16 | Avg Loss: 0.0645
Epoch 17 | Avg Loss: 0.0401
Epoch 18 | Avg Loss: 0.0458
Epoch 19 | Avg Loss: 0.0491
Epoch 20 | Avg Loss: 0.0271


In [10]:
torch.save(model.state_dict(), "mmcto_phaseII_gaiting.pth")

In [11]:
from tqdm import tqdm
print(f"\nEvaluating on Phase {PHASE} Test Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase II Test Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [12]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      66.86  74.37  56.41
summarization              60.82  62.74  50.22
smiles                     61.09   2.57  50.39
description                63.42  75.01  55.39
criteria                   60.91  70.67  48.80
enrollment                 61.59  76.23  50.00
diseases                   60.41  42.55  49.00
drugs                      61.71  60.29  50.42
All (Fused)                66.92  74.37  55.54


## PHASE III

In [12]:
print(f"\nTraining MMCTO on Phase {PHASE} with ADMET features...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase III with ADMET features...

Epoch 1 | Avg Loss: 0.4003
Epoch 2 | Avg Loss: 0.3999
Epoch 3 | Avg Loss: 0.3918
Epoch 4 | Avg Loss: 0.3757
Epoch 5 | Avg Loss: 0.3572
Epoch 6 | Avg Loss: 0.3145
Epoch 7 | Avg Loss: 0.2685
Epoch 8 | Avg Loss: 0.2027
Epoch 9 | Avg Loss: 0.1504
Epoch 10 | Avg Loss: 0.1277
Epoch 11 | Avg Loss: 0.0881
Epoch 12 | Avg Loss: 0.0608
Epoch 13 | Avg Loss: 0.0428
Epoch 14 | Avg Loss: 0.0406
Epoch 15 | Avg Loss: 0.0658
Epoch 16 | Avg Loss: 0.0203
Epoch 17 | Avg Loss: 0.0269
Epoch 18 | Avg Loss: 0.0209
Epoch 19 | Avg Loss: 0.0169
Epoch 20 | Avg Loss: 0.0176


In [13]:
torch.save(model.state_dict(), "mmcto_phaseIII_gating.pth")

In [14]:
print(f"\nEvaluating on Phase {PHASE} Test Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase III Test Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [15]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      71.58   0.84  47.79
summarization              74.78  84.10  52.60
smiles                     75.67  44.38  52.22
description                71.76   5.25  50.65
criteria                   69.21  32.80  46.09
enrollment                 72.56  84.10  50.00
diseases                   69.05  25.17  43.70
drugs                      73.91  19.38  51.98
All (Fused)                74.13  81.78  51.70


# CTOD

### PHASE I

In [12]:
print(f"\nTraining MMCTO on Phase {PHASE} with ADMET features...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase I with ADMET features...

Epoch 1 | Avg Loss: 0.4314
Epoch 2 | Avg Loss: 0.4071
Epoch 3 | Avg Loss: 0.4024
Epoch 4 | Avg Loss: 0.3904
Epoch 5 | Avg Loss: 0.3772
Epoch 6 | Avg Loss: 0.3657
Epoch 7 | Avg Loss: 0.3386
Epoch 8 | Avg Loss: 0.3071
Epoch 9 | Avg Loss: 0.2793
Epoch 10 | Avg Loss: 0.2519
Epoch 11 | Avg Loss: 0.2398
Epoch 12 | Avg Loss: 0.2040
Epoch 13 | Avg Loss: 0.1744
Epoch 14 | Avg Loss: 0.1576
Epoch 15 | Avg Loss: 0.1521
Epoch 16 | Avg Loss: 0.1305
Epoch 17 | Avg Loss: 0.1138
Epoch 18 | Avg Loss: 0.1017
Epoch 19 | Avg Loss: 0.1069
Epoch 20 | Avg Loss: 0.0857


In [14]:
torch.save(model.state_dict(), "mmcto_phaseI_CTOD_gating.pth")

In [15]:
print(f"\nEvaluating on Phase {PHASE} Valid Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase I Valid Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [16]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      76.37  88.20  45.05
summarization              83.86  60.49  58.78
smiles                     79.30  85.39  49.79
description                79.10  32.73  50.58
criteria                   81.27  87.03  53.26
enrollment                 64.03   0.00  20.52
diseases                   81.47  29.11  54.02
drugs                      78.15   2.99  45.32
All (Fused)                91.35  88.58  78.38


### PHASE II

In [11]:
print(f"\nTraining MMCTO on Phase {PHASE} with ADMET features...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase II with ADMET features...

Epoch 1 | Avg Loss: 0.4267
Epoch 2 | Avg Loss: 0.4122
Epoch 3 | Avg Loss: 0.4068
Epoch 4 | Avg Loss: 0.3973
Epoch 5 | Avg Loss: 0.3836
Epoch 6 | Avg Loss: 0.3734
Epoch 7 | Avg Loss: 0.3479
Epoch 8 | Avg Loss: 0.3353
Epoch 9 | Avg Loss: 0.3126
Epoch 10 | Avg Loss: 0.3003
Epoch 11 | Avg Loss: 0.2820
Epoch 12 | Avg Loss: 0.2719
Epoch 13 | Avg Loss: 0.2525
Epoch 14 | Avg Loss: 0.2366
Epoch 15 | Avg Loss: 0.2233
Epoch 16 | Avg Loss: 0.2215
Epoch 17 | Avg Loss: 0.2112
Epoch 18 | Avg Loss: 0.1957
Epoch 19 | Avg Loss: 0.1813
Epoch 20 | Avg Loss: 0.1696


In [12]:
torch.save(model.state_dict(), "mmcto_phaseII_CTOD_gating.pth")

In [13]:
print(f"\nEvaluating on Phase {PHASE} Valid Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase II Valid Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [14]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      78.41  83.48  49.80
summarization              76.49  70.67  47.54
smiles                     79.78  87.43  52.87
description                76.12  52.18  47.43
criteria                   76.16   0.00  48.13
enrollment                 62.17  64.83  17.76
diseases                   78.40  21.07  50.09
drugs                      79.98  14.99  52.59
All (Fused)                93.12  91.06  83.54


### PHASE III

In [12]:
print(f"\nTraining MMCTO on Phase {PHASE}...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase III...

Epoch 1 | Avg Loss: 0.3603
Epoch 2 | Avg Loss: 0.3451
Epoch 3 | Avg Loss: 0.3296
Epoch 4 | Avg Loss: 0.3155
Epoch 5 | Avg Loss: 0.2892
Epoch 6 | Avg Loss: 0.2633
Epoch 7 | Avg Loss: 0.2267
Epoch 8 | Avg Loss: 0.2013
Epoch 9 | Avg Loss: 0.1860
Epoch 10 | Avg Loss: 0.1641
Epoch 11 | Avg Loss: 0.1501
Epoch 12 | Avg Loss: 0.1325
Epoch 13 | Avg Loss: 0.1150
Epoch 14 | Avg Loss: 0.1166
Epoch 15 | Avg Loss: 0.1048
Epoch 16 | Avg Loss: 0.0974
Epoch 17 | Avg Loss: 0.0904
Epoch 18 | Avg Loss: 0.0829
Epoch 19 | Avg Loss: 0.1002
Epoch 20 | Avg Loss: 0.0896


In [13]:
torch.save(model.state_dict(), "mmcto_phaseIII_CTOD.pth")

In [14]:
from tqdm import tqdm
import numpy as np

print(f"\nEvaluating on Phase {PHASE} Valid Set...\n")
model.eval()

# Prepare prediction containers
all_preds = {
    "fused": [],
    "target": [],
}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating", leave=True):
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        # Store fused and target predictions
        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())
        else:
            print("[Warning] No fused prediction returned.")

        # Store modality-specific predictions
        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())
            else:
                print(f"[Warning] No prediction for modality: {mod}")

# Safely concatenate arrays
for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        print(f"[Warning] No data to concatenate for: {key}")
        all_preds[key] = np.array([])  # or skip if not needed



Evaluating on Phase III Valid Set...



Evaluating:   0%|                                                                               | 0/33 [00:00<?, ?it/s]C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(
Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 33/33 [00:14<00:00,  2.25it/s]


In [15]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      84.98  64.76  55.38
summarization              79.99  66.58  45.15
smiles                     81.88  81.27  49.14
description                82.74  89.52  49.91
criteria                   80.18  83.67  47.41
enrollment                 91.09  90.24  76.64
diseases                   79.31  59.10  41.97
drugs                      83.46  63.79  51.95
All (Fused)                89.05  91.03  72.59


In [16]:
model.load_state_dict(torch.load("C:/Users/Carol/Documents/Data Code and Deliverables/Data Group Part/project/models/mmcto_phaseII.pth"))
model.eval()
model.to(DEVICE)

C:\Users\Carol\AppData\Local\Temp\ipykernel_13708\3584377234.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("C:/Users/Carol/Documents/D

RuntimeError: Error(s) in loading state_dict for MMCTO:
	size mismatch for gate_fc.weight: copying a param with shape torch.Size([8, 1536]) from checkpoint, the shape in current model is torch.Size([8, 6144]).

In [ ]:
test_dataset = HINTDataset(
    data_prefix=data_prefix,
    ann_file_name=f"phase_III_valid",  # or whatever split you want
    serialize_data=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=test_dataset.collate_fn,
)

In [20]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# 🧠 Setup — make sure model, DEVICE, test_loader are already defined
model.eval()

# 🔹 Prediction containers
all_preds = {"fused": [], "target": []}
indexes = []  # collect batch indices
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

# 🔁 Run inference
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]
        idxs = batch["idx"].cpu().numpy()  # <-- collect idx

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())
            indexes.append(idxs)

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

# ✅ Prepare for insertion
all_preds["fused"] = [np.atleast_1d(p) for p in all_preds["fused"]]
indexes = [np.atleast_1d(i) for i in indexes]
fused_preds = np.concatenate(all_preds["fused"])
indexes = np.concatenate(indexes)

# 🔒 Sanity check
assert len(fused_preds) == len(indexes), "Mismatch between predictions and indexes"

# 📂 Path to your test CSV
csv_path = r"C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint\phase_III_valid.csv"

# 📖 Load the CSV
df = pd.read_csv(csv_path)

# ✏️ Insert predictions into the correct rows
df.loc[indexes, "fused_pred"] = fused_preds

# 💾 Save updated CSV
df.to_csv(csv_path, index=False)
print(f"✅ Added {len(fused_preds)} predictions to CSV:\n{csv_path}")


Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 43/43 [00:19<00:00,  2.22it/s]

✅ Added 676 predictions to CSV:
C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint\phase_III_valid.csv
